In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path(r"E:\Github\Projects\Langchain-Learning\.env")

print("File exists:", env_path.is_file())

load_dotenv(env_path, override=True)

key = os.getenv("OPENROUTER_API_KEY")
print("API key loaded:", bool(key and key.strip()))  

File exists: True
API key loaded: True


### 1.Make the model

In [2]:
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(model="nvidia/nemotron-3-ultra-550b-a55b:free")

In [3]:
model

ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6', 'langchain-openrouter': '0.2.9'}}, profile={'name': 'Nemotron 3 Ultra (free)', 'release_date': '2026-06-04', 'last_updated': '2026-06-04', 'open_weights': True, 'max_input_tokens': 1000000, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': False, 'attachment': False, 'temperature': True, 'tool_call_streaming': True}, client=<openrouter.sdk.OpenRouter object at 0x00000230A7A54980>, openrouter_api_key=SecretStr('**********'), app_url='https://docs.langchain.com', app_title='LangChain', model_name='nvidia/nemotron-3-ultra-550b-a55b:free', model_kwargs={})

### 2. Ask it something
##### Invoke sends the text and returns the reply. That's the first AI program

In [4]:
reply = model.invoke("What is LangChain, in one sentence?")
reply.text

'LangChain is an open-source framework designed to simplify the creation of applications powered by large language models (LLMs) by providing modular components, chains, and agents for prompt management, memory, and external data integration.'

In [5]:
reply.content

'LangChain is an open-source framework designed to simplify the creation of applications powered by large language models (LLMs) by providing modular components, chains, and agents for prompt management, memory, and external data integration.'

### 3. The reply is an object, not a string

In [6]:
reply

AIMessage(content='LangChain is an open-source framework designed to simplify the creation of applications powered by large language models (LLMs) by providing modular components, chains, and agents for prompt management, memory, and external data integration.', additional_kwargs={'reasoning_content': 'The user wants a one-sentence definition of LangChain.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user wants a one-sentence definition of LangChain.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1792554588-TWL4seARyFQ8HPkJRMNN', 'created': 1792554588, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01a15c94-54c6-752a-9d01-e48e9cba4f3c-0', tool_calls=[], invalid_tool_calls=[], usa

### 4. What the call cost?

In [7]:
reply.usage_metadata

{'input_tokens': 25,
 'output_tokens': 58,
 'total_tokens': 83,
 'input_token_details': {'cache_read': 0, 'cache_creation': 0},
 'output_token_details': {'reasoning': 14}}

### 5. Which model answered, and what it cost?

In [8]:
reply.response_metadata  

{'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free',
 'id': 'gen-1792554588-TWL4seARyFQ8HPkJRMNN',
 'created': 1792554588,
 'object': 'chat.completion',
 'finish_reason': 'stop',
 'logprobs': None,
 'model_provider': 'openrouter',
 'cost': 0.0,
 'cost_details': {'upstream_inference_completions_cost': 0.0,
  'upstream_inference_prompt_cost': 0.0,
  'upstream_inference_cost': 0.0}}

### 6. Asking again to get something different

In [13]:
model.invoke("Give me one interesting fact about the moon.").text

"**The Moon is slowly drifting away from Earth at a rate of about 1.5 inches (3.8 cm) per year.**\n\nThis happens because of tidal friction: the Moon's gravity pulls on Earth's oceans, creating a tidal bulge. Because Earth rotates faster than the Moon orbits, that bulge gets dragged slightly ahead of the Moon. The bulge's gravity then pulls the Moon forward, giving it an energy boost that pushes it into a higher, slower orbit."

In [ ]:
# "The Moon is **drifting away from Earth at a rate of about 3.8 centimeters (1.5 inches) per year**.\n\nThis happens because of tidal friction: the Moon's gravity pulls on Earth's oceans, creating a tidal bulge. Because Earth rotates faster than the Moon orbits, that bulge gets pulled slightly ahead of the Moon. The bulge's gravity then pulls the Moon forward, boosting its orbital energy and pushing it into a higher, slower orbit."

The answers differ. A model predicts the next word with deliberate randomness, ir doesn't look one up.


 ***Same question,different answer** is noemal,not a bug

### 7. Turning the randomness down

In [14]:
steady = ChatOpenRouter(model="nvidia/nemotron-3-ultra-550b-a55b:free", temperature = 0)

steady.invoke("Name the largest planet in our solar system.").text

'The largest planet in our solar system is **Jupiter**.'

 Run twice: the answer should be the same or very close. Use temperature=0 for facts and tests, higher for ideas and variety.

 Even at zero, runs aren't guaranteed identical. Treat it as "as steady as it gets", not a promise.

#### 8. A limit that fails in as way you would not guess

In [16]:
clipped = ChatOpenRouter(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    temperature = 0,
    max_tokens=20,
)

answer = clipped.invoke("Explain how a jet engine works.")
answer.text

'The user wants an explanation of how a jet engine works.\nI should structure this clearly:\n1.'

In [17]:
answer

AIMessage(content='The user wants an explanation of how a jet engine works.\nI should structure this clearly:\n1.', additional_kwargs={'reasoning_content': 'The user wants an explanation of how a jet engine works.\nI should structure this clearly:\n1.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user wants an explanation of how a jet engine works.\nI should structure this clearly:\n1.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1792562414-B3TKfmzBKTPiDNiQMAjQ', 'created': 1792562414, 'object': 'chat.completion', 'finish_reason': 'length', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01a15d0b-c15c-716e-9787-bc9626fbb064-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_tokens': 20, 'total_toke

In [19]:
answer.response_metadata["finish_reason"]

'length'

In [20]:
answer.usage_metadata

{'input_tokens': 24,
 'output_tokens': 20,
 'total_tokens': 44,
 'input_token_details': {'cache_read': 0, 'cache_creation': 0},
 'output_token_details': {'reasoning': 24}}